# Film Format Checker and Converter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadelh/media-compression-tools/blob/main/film_format_converter.ipynb)

An online film festival receives films in all kinds of formats, and every film has to be delivered in one exact format. This notebook automates the check:

1. **Probe** every film with `ffprobe` and read its container, codecs, frame rate, resolution, bitrates and audio channels.
2. **Audit** each film against the festival's specification and write a plain-text report that lists the films that fail and the problem fields.
3. **Convert** only the failing films with `ffmpeg`, saving each as `<name>_formatOK.mp4`.
4. **Re-check** the converted films and write a second report.

The specification the films must meet:

| Field | Required |
|---|---|
| Container | mp4 |
| Video codec | H.264 |
| Audio codec | AAC |
| Frame rate | 25 fps |
| Aspect ratio / resolution | 16:9 / 640 x 360 |
| Video bitrate | 2 to 5 Mb/s |
| Audio bitrate | up to 256 kb/s |
| Audio channels | stereo |

## 1. Setup and paths

In [1]:
import json, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path
import pandas as pd

# The five film excerpts are large (the raw AVI alone is 222 MB), so they are published as a release asset
FILMS_URL = "https://github.com/ziadelh/media-compression-tools/releases/download/data-v1/films.zip"
if not Path("films").is_dir():
    print("Downloading the film excerpts (about 160 MB)...")
    urllib.request.urlretrieve(FILMS_URL, "films.zip")
    zipfile.ZipFile("films.zip").extractall("films")
    os.remove("films.zip")

# Directories
IN_DIR  = Path("films").resolve()
OUT_DIR = Path("./converted").resolve()
REPORTS = Path("./reports").resolve()
for d in (OUT_DIR, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

# Executables (Colab has both; on a PC install FFmpeg and add it to the PATH)
FFPROBE = shutil.which("ffprobe")
FFMPEG  = shutil.which("ffmpeg")
if not (FFPROBE and FFMPEG):
    raise RuntimeError("ffmpeg and ffprobe were not found. Install FFmpeg (https://ffmpeg.org/download.html) and add it to the PATH.")

# run commands quietly and return result
def run(cmd: str):
    return subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# environment check
print(run(f'"{FFPROBE}" -version').stdout.splitlines()[0])
print(run(f'"{FFMPEG}" -version').stdout.splitlines()[0])

# Discover input files
inputs = sorted(p for p in IN_DIR.glob("*") if p.suffix.lower() in {".mp4",".mov",".m4v",".avi",".mkv",".webm"})
print(f"Found {len(inputs)} input file(s).")

ffprobe version 7.1.1-essentials_build-www.gyan.dev Copyright (c) 2007-2025 the FFmpeg developers
ffmpeg version 7.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Found 5 input file(s).


## 2. Probing the media metadata

`probe()` runs `ffprobe` and turns its JSON output into a `MediaMeta` record that holds the container, the first video stream and the first audio stream.

In [2]:
from dataclasses import dataclass
from typing import Optional

def fps_from_rate(rate: Optional[str]) -> Optional[float]:
    # Convert ffprobe rate string to float fps.
    if not rate or rate in {"0/0", "0"}:
        return None
    if "/" in rate:
        n, d = rate.split("/")
        try:
            d = float(d)
            return float(n) / d if d != 0 else None
        except Exception:
            return None
    try:
        return float(rate)
    except Exception:
        return None

def human_bps(bps: Optional[int]) -> str:
    # Format bitrate values into readable units
    if not bps or bps <= 0:
        return "unknown"
    units = ["b/s", "kb/s", "Mb/s", "Gb/s"]
    x = float(bps)
    for u in units:
        if x < 1000:
            return f"{x:.0f} {u}"
        x /= 1000
    return f"{x:.1f} Tb/s"

@dataclass
class MediaMeta:
    path: Path
    container_names: str
    container_long: Optional[str]
    major_brand: Optional[str]
    duration: Optional[float]
    # video
    vcodec: Optional[str]
    width: Optional[int]
    height: Optional[int]
    fps: Optional[float]
    sar: Optional[str]
    dar: Optional[str]
    v_bitrate: Optional[int]
    # audio
    acodec: Optional[str]
    channels: Optional[int]
    a_bitrate: Optional[int]
    has_audio: bool

def probe(path: Path) -> MediaMeta:
    # Run ffprobe on a file and return its metadata as a MediaMeta object
    cp = run(f'"{FFPROBE}" -v error -print_format json -show_streams -show_format "{path}"')
    info = json.loads(cp.stdout)

    fmt = info.get("format", {}) or {}
    v   = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)
    a   = next((s for s in info.get("streams", []) if s.get("codec_type") == "audio"), None)

    return MediaMeta(
        path=path,
        container_names = fmt.get("format_name", ""),
        container_long  = fmt.get("format_long_name"),
        major_brand     = (fmt.get("tags") or {}).get("major_brand"),
        duration        = float(fmt["duration"]) if "duration" in fmt else None,
        vcodec          = (v or {}).get("codec_name"),
        width           = (v or {}).get("width"),
        height          = (v or {}).get("height"),
        fps             = fps_from_rate((v or {}).get("avg_frame_rate") or (v or {}).get("r_frame_rate")),
        sar             = (v or {}).get("sample_aspect_ratio"),
        dar             = (v or {}).get("display_aspect_ratio"),
        v_bitrate       = int((v or {}).get("bit_rate")) if (v or {}).get("bit_rate") else (int(fmt["bit_rate"]) if fmt.get("bit_rate") else None),
        acodec          = (a or {}).get("codec_name"),
        channels        = (a or {}).get("channels"),
        a_bitrate       = int((a or {}).get("bit_rate")) if (a or {}).get("bit_rate") else None,
        has_audio       = a is not None
    )

## 3. Validation against the specification

`validate()` compares one file's metadata with the festival specification and returns whether it passes, plus a short reason for every field that fails. `_is_mp4()` recognises the whole MP4 family, because ffprobe reports `mov,mp4,m4a,3gp,3g2,mj2` for an MP4 file.

In [3]:
SPEC = {
    "fps": 25.0,
    "wh": (640, 360),
    "aspect": (16, 9),
    "vcodec": {"h264", "avc1"},
    "acodec": "aac",
    "vbit_min": 2_000_000,
    "vbit_max": 5_000_000,
    "abits_max": 256_000,
    "channels": 2
}

FPS_TOL = 0.02
AR_TOL  = 0.015

def _is_mp4(meta: MediaMeta) -> bool:
    # Return True if container is recognized as MP4 family
    names = {x.strip().lower() for x in (meta.container_names or "").split(",")}
    if "mp4" in names:
        return True
    if (meta.major_brand or "").lower() in {"isom", "mp41", "mp42", "iso2"}:
        return True
    if meta.container_long and "mp4" in meta.container_long.lower():
        return True
    return False

def validate(m: MediaMeta) -> tuple[bool, list[str]]:
    # Validate one file's metadata against the festival SPEC and return ok, reasons.
    fails = []

    # MP4 detection
    if not _is_mp4(m):
        fails.append(f"container: expected mp4, found {m.container_names or m.container_long or 'unknown'}")

    # Video codec
    if (m.vcodec or "").lower() not in SPEC["vcodec"]:
        fails.append(f"video codec: expected h.264, found {m.vcodec or 'unknown'}")

    # Audio presence and codec must be AAC
    if not m.has_audio:
        fails.append("audio: expected AAC stereo, found none")
    elif (m.acodec or "").lower() != SPEC["acodec"]:
        fails.append(f"audio codec: expected aac, found {m.acodec or 'unknown'}")

    # Exact resolution
    if (m.width, m.height) != SPEC["wh"]:
        fails.append(f"resolution: expected 640x360, found {m.width}x{m.height}")

    # Frame rate
    if (m.fps is None) or abs(m.fps - SPEC["fps"]) > FPS_TOL:
        fails.append(f"frame rate: expected 25.000, found {m.fps}")

    # Aspect ratio
    target = SPEC["aspect"][0] / SPEC["aspect"][1]
    r = (m.width / m.height) if (m.width and m.height) else None
    if r is None or abs(r - target) > AR_TOL:
        fails.append("aspect ratio: expected 16:9")

    # Video bitrate
    if (not m.v_bitrate) or not (SPEC["vbit_min"] <= m.v_bitrate <= SPEC["vbit_max"]):
        fails.append(f"video bitrate: expected 2–5 Mb/s, found {human_bps(m.v_bitrate)}")

    # Audio bitrate
    if m.a_bitrate and m.a_bitrate > SPEC["abits_max"]:
        fails.append(f"audio bitrate: expected ≤ 256 kb/s, found {human_bps(m.a_bitrate)}")

    # Stereo channels
    if (m.channels or 0) != SPEC["channels"]:
        fails.append(f"audio channels: expected stereo (2), found {m.channels or 'unknown'}")

    return (len(fails) == 0, fails)

## 4. Pre-conversion audit

Every film is probed and validated. The result is written to a plain-text report and shown here.

In [4]:
from datetime import datetime

# Timestamp for report filenames
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Collect validation results
pre_rows = []
for p in inputs:
    m = probe(p)
    ok, reasons = validate(m)
    pre_rows.append([p.name, "PASS" if ok else "FAIL", "; ".join(reasons)])


# Save pre conversion TXT report
pre_txt = REPORTS / f"pre_conversion_audit_{ts}.txt"
with pre_txt.open("w", encoding="utf-8") as f:
    f.write("Narbonne Festival — Pre-Conversion Audit Report\n")
    for name, status, notes in pre_rows:
        f.write(f"{name}: {status}\n")
        if status == "FAIL" and notes:
            for n in notes.split("; "):
                f.write(f"  - {n}\n")
            f.write("\n")

print("Saved pre-audit report:", pre_txt.name)
print()
print(pre_txt.read_text(encoding="utf-8"))

Saved pre-audit report: pre_conversion_audit_20261005_031403.txt

Narbonne Festival — Pre-Conversion Audit Report
Cosmos_War_of_the_Planets.mp4: FAIL
  - resolution: expected 640x360, found 628x354
  - frame rate: expected 25.000, found 29.97002997002997
  - audio bitrate: expected ≤ 256 kb/s, found 317 kb/s

Last_man_on_earth_1964.mov: FAIL
  - video codec: expected h.264, found prores
  - audio codec: expected aac, found pcm_s16le
  - frame rate: expected 25.000, found 23.976023976023978
  - video bitrate: expected 2–5 Mb/s, found 9 Mb/s
  - audio bitrate: expected ≤ 256 kb/s, found 2 Mb/s

The_Gun_and_the_Pulpit.avi: FAIL
  - container: expected mp4, found avi
  - video codec: expected h.264, found rawvideo
  - audio codec: expected aac, found pcm_s16le
  - resolution: expected 640x360, found 720x404
  - video bitrate: expected 2–5 Mb/s, found 87 Mb/s
  - audio bitrate: expected ≤ 256 kb/s, found 2 Mb/s

The_Hill_Gang_Rides_Again.mp4: FAIL
  - video bitrate: expected 2–5 Mb/s, found

## 5. Converting the films that fail

`build_cmd()` builds an `ffmpeg` command that forces the specification: H.264 video scaled to 640 x 360 with a square pixel aspect ratio, 25 fps, about 3 Mb/s (capped at 5), and AAC stereo audio at 192 kb/s. A film with no audio track gets a silent stereo track so it still meets the spec. Films that already pass are left alone.

In [5]:
def build_cmd(meta: MediaMeta, out_path: Path) -> str:
    # Enforce spec
    vf = 'scale=640:360,setsar=1,setdar=16/9'
    common = f'-c:v libx264 -pix_fmt yuv420p -r 25 -b:v 3M -maxrate 5M -bufsize 6M -vf "{vf}" -movflags +faststart'
    if meta.has_audio:
        return f'"{FFMPEG}" -y -i "{meta.path}" {common} -c:a aac -ac 2 -b:a 192k -f mp4 "{out_path}"'
    else:
        return (
            f'"{FFMPEG}" -y -i "{meta.path}" '
            f'-f lavfi -i anullsrc=channel_layout=stereo:sample_rate=48000 -shortest '
            f'{common} -c:a aac -ac 2 -b:a 192k -f mp4 "{out_path}"'
        )

# Convert only non compliant inputs
converted = []
for p in inputs:
    meta = probe(p)
    ok, _ = validate(meta)
    if ok:
        continue
    out_path = OUT_DIR / f"{p.stem}_formatOK.mp4"
    proc = run(build_cmd(meta, out_path))
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr)
    converted.append(out_path)
    print("Converted:", out_path.name)

Converted: Cosmos_War_of_the_Planets_formatOK.mp4


Converted: Last_man_on_earth_1964_formatOK.mp4


Converted: The_Gun_and_the_Pulpit_formatOK.mp4


Converted: The_Hill_Gang_Rides_Again_formatOK.mp4


Converted: Voyage_to_the_Planet_of_Prehistoric_Women_formatOK.mp4


## 6. Post-conversion audit

In [6]:
# Post conversion audit
targets = converted or sorted(OUT_DIR.glob("*_formatOK.mp4"))
post_rows = []
for out in targets:
    m2 = probe(out)
    ok2, reasons2 = validate(m2)
    post_rows.append([out.name, "PASS" if ok2 else "FAIL", "; ".join(reasons2)])

# final table
final_df = pd.DataFrame(post_rows, columns=["File", "Status", "Notes"])[["File", "Status"]]
display(final_df)

# Save post conversion TXT
post_txt = REPORTS / f"post_conversion_audit_{ts}.txt"
with post_txt.open("w", encoding="utf-8") as f:
    f.write("Narbonne Festival — Post-Conversion Audit Report\n")
    f.write("All converted files must PASS the spec.\n\n")
    for name, status, notes in post_rows:
        f.write(f"{name}: {status}\n")
        if notes:
            for n in notes.split("; "):
                f.write(f"  - {n}\n")
            f.write("\n")

print("Saved post-audit report:", post_txt.name)

,File,Status
0,Cosmos_War_of_the_Planets_formatOK.mp4,PASS
1,Last_man_on_earth_1964_formatOK.mp4,PASS
2,The_Gun_and_the_Pulpit_formatOK.mp4,PASS
3,The_Hill_Gang_Rides_Again_formatOK.mp4,PASS
4,Voyage_to_the_Planet_of_Prehistoric_Women_form...,PASS


Saved post-audit report: post_conversion_audit_20261005_031403.txt


### The converted films, field by field

In [7]:
rows = []
for out in targets:
    m = probe(out)
    rows.append({
        "File": out.name,
        "Container": "mp4" if _is_mp4(m) else m.container_names,
        "Video": m.vcodec,
        "Audio": m.acodec,
        "Resolution": f"{m.width}x{m.height}",
        "FPS": round(m.fps, 3) if m.fps else None,
        "Video bitrate": human_bps(m.v_bitrate),
        "Audio bitrate": human_bps(m.a_bitrate),
        "Channels": m.channels,
    })
pd.DataFrame(rows)

,File,Container,Video,Audio,Resolution,FPS,Video bitrate,Audio bitrate,Channels
0,Cosmos_War_of_the_Planets_formatOK.mp4,mp4,h264,aac,640x360,25.0,3 Mb/s,193 kb/s,2
1,Last_man_on_earth_1964_formatOK.mp4,mp4,h264,aac,640x360,25.0,3 Mb/s,192 kb/s,2
2,The_Gun_and_the_Pulpit_formatOK.mp4,mp4,h264,aac,640x360,25.0,3 Mb/s,193 kb/s,2
3,The_Hill_Gang_Rides_Again_formatOK.mp4,mp4,h264,aac,640x360,25.0,3 Mb/s,191 kb/s,2
4,Voyage_to_the_Planet_of_Prehistoric_Women_form...,mp4,h264,aac,640x360,25.0,3 Mb/s,192 kb/s,2
